In [1]:
import os
import subprocess
import tensorflow as tf
from pathlib import Path

I0000 00:00:1780180729.081195     711 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780180729.378023     711 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780180730.809580     711 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# ============================================================
# Tool paths (macro-like constants)
# ============================================================
MLIR_OPT = "/workspace/PyTorchSim/Tensorflow/binaries/mlir-opt"
MLIR_TRANSLATE = "/workspace/PyTorchSim/Tensorflow/binaries/mlir-translate"
MLIR_OPT_PYTORCHSIM = "/riscv-llvm/bin/mlir-opt"
STABLEHLO_OPT = "/workspace/PyTorchSim/Tensorflow/binaries/stablehlo-opt"
STABLEHLO_TRANSLATE = "/workspace/PyTorchSim/Tensorflow/binaries/stablehlo-translate"

OUT_DIR = "out"
os.makedirs(OUT_DIR, exist_ok=True)
def run(cmd):
    print(">>", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

In [3]:
# ============================================================
# 0. Target TensorFlow function
# ============================================================
@tf.function(jit_compile=True)
def add_fn(x, y):
    return x + y

x = tf.constant([1.0, 2.0], dtype=tf.float32)
y = tf.constant([3.0, 4.0], dtype=tf.float32)


E0000 00:00:1780180732.267754     711 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
# ============================================================
# 1. HLO
# ============================================================
ir = add_fn.experimental_get_compiler_ir(x, y)(stage="hlo")
print("=== HLO ===")
print(ir)


=== HLO ===
HloModule a_inference_add_fn_8__.1, entry_computation_layout={(f32[2]{0}, f32[2]{0})->f32[2]{0}}

ENTRY %a_inference_add_fn_8__.1 (arg0.1: f32[2], arg1.1: f32[2]) -> f32[2] {
  %arg0.1 = f32[2]{0} parameter(0), parameter_replication={false}, metadata={op_name="XLA_Args"}
  %reshape.2 = f32[2]{0} reshape(f32[2]{0} %arg0.1)
  %arg1.1 = f32[2]{0} parameter(1), parameter_replication={false}, metadata={op_name="XLA_Args"}
  %reshape.3 = f32[2]{0} reshape(f32[2]{0} %arg1.1)
  %add.1 = f32[2]{0} add(f32[2]{0} %reshape.2, f32[2]{0} %reshape.3), metadata={op_type="AddV2" op_name="add" source_file="/opt/conda/lib/python3.11/site-packages/tensorflow/python/framework/ops.py" source_line=1221}
  %XLA_Retvals.3 = f32[2]{0} reshape(f32[2]{0} %add.1), metadata={op_name="XLA_Retvals"}
  %XLA_Retvals.4 = (f32[2]{0}) tuple(f32[2]{0} %XLA_Retvals.3), metadata={op_name="XLA_Retvals"}
  ROOT %XLA_Retvals.5 = f32[2]{0} get-tuple-element((f32[2]{0}) %XLA_Retvals.4), index=0, metadata={op_name="XLA

I0000 00:00:1780180732.365254     711 service.cc:153] XLA service 0x558f131510b0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780180732.365277     711 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)


In [5]:
# ============================================================
# 2. TF Dialect MLIR
# ============================================================
def tf_function_to_tf_mlir(fn):
    concrete_fn = fn.get_concrete_function(
        tf.TensorSpec([2], tf.float32),
        tf.TensorSpec([2], tf.float32),
    )
    return tf.mlir.experimental.convert_function(
        concrete_fn,
        pass_pipeline="tf-standard-pipeline",
        show_debug_info=False,
    )



tf_mlir = tf_function_to_tf_mlir(add_fn)
tf_mlir_path = f"{OUT_DIR}/tf_dialect.mlir"

with open(tf_mlir_path, "w") as f:
    f.write(tf_mlir)

print("=== TF DIALECT MLIR ===")
print(tf_mlir)


=== TF DIALECT MLIR ===
module attributes {tf.versions = {bad_consumers = [], min_consumer = 0 : i32, producer = 2474 : i32}} {
  func.func @__inference_add_fn_8(%arg0: tensor<2xf32> {tf._user_specified_name = "x"}, %arg1: tensor<2xf32> {tf._user_specified_name = "y"}) -> tensor<2xf32> attributes {allow_soft_placement = false, tf.entry_function = {control_outputs = "", inputs = "x,y", outputs = "identity_RetVal"}} {
    %0 = "tf.AddV2"(%arg0, %arg1) {device = ""} : (tensor<2xf32>, tensor<2xf32>) -> tensor<2xf32>
    %1 = "tf.Identity"(%0) {device = ""} : (tensor<2xf32>) -> tensor<2xf32>
    return %1 : tensor<2xf32>
  }
}



In [6]:
# ============================================================
# 3. STABLEHLO Dialect MLIR
# ============================================================
ir = add_fn.experimental_get_compiler_ir(x, y)(stage="stablehlo")
input_mlir = Path(OUT_DIR) / "00_input_stablehlo.mlir"
input_mlir.write_text(ir)

print("=== STABLEHLO MLIR ===")
print(input_mlir.read_text())

=== STABLEHLO MLIR ===
#loc1 = loc("XLA_Args")
module @a_inference_add_fn_8__.1 attributes {mhlo.cross_program_prefetches = [], mhlo.input_output_alias = [], mhlo.is_dynamic = false, mhlo.use_auto_spmd_partitioning = false} {
  func.func @main(%arg0: tensor<2xf32> loc("XLA_Args"), %arg1: tensor<2xf32> loc("XLA_Args")) -> tensor<2xf32> {
    %0 = stablehlo.reshape %arg0 : (tensor<2xf32>) -> tensor<2xf32> loc(#loc2)
    %1 = stablehlo.reshape %arg1 : (tensor<2xf32>) -> tensor<2xf32> loc(#loc3)
    %2 = stablehlo.add %0, %1 : tensor<2xf32> loc(#loc7)
    %3 = stablehlo.reshape %2 : (tensor<2xf32>) -> tensor<2xf32> loc(#loc6)
    return %3 : tensor<2xf32> loc(#loc)
  } loc(#loc)
} loc(#loc)
#loc = loc(unknown)
#loc2 = loc("reshape.2")
#loc3 = loc("reshape.3")
#loc4 = loc("add")
#loc5 = loc("/opt/conda/lib/python3.11/site-packages/tensorflow/python/framework/ops.py":1221:0)
#loc6 = loc("XLA_Retvals")
#loc7 = loc(fused[#loc4, #loc5])



In [7]:
out1 = Path(OUT_DIR) / "01_stablehlo_clean.mlir"

run([
    STABLEHLO_OPT,
    str(input_mlir),
    "--stablehlo-target-independent-optimization",
    "-o", str(out1),
])

print("=== STABLEHLO MLIR (Clean)===")
print(out1.read_text())


>> /workspace/PyTorchSim/Tensorflow/binaries/stablehlo-opt out/00_input_stablehlo.mlir --stablehlo-target-independent-optimization -o out/01_stablehlo_clean.mlir
=== STABLEHLO MLIR (Clean)===
module @a_inference_add_fn_8__.1 attributes {mhlo.cross_program_prefetches = [], mhlo.input_output_alias = [], mhlo.is_dynamic = false, mhlo.use_auto_spmd_partitioning = false} {
  func.func @main(%arg0: tensor<2xf32>, %arg1: tensor<2xf32>) -> tensor<2xf32> {
    %0 = stablehlo.add %arg0, %arg1 : tensor<2xf32>
    return %0 : tensor<2xf32>
  }
}




In [8]:
# ============================================================
# 4. StableHLO → Linalg (Tensor)
# ============================================================
out2_1 = Path(OUT_DIR) / "02_linalg_generic_tensor.mlir"

run([
    STABLEHLO_OPT,
    str(out1),
    "--stablehlo-legalize-to-linalg",
    "-linalg-fuse-elementwise-ops",  
    # "--linalg-specialize-generic-ops",  
    "--stablehlo-target-independent-optimization",
    "-o", str(out2_1),
])

print("=== GNERIC LINALG (TENSOR) MLIR ===")
print(out2_1.read_text())

>> /workspace/PyTorchSim/Tensorflow/binaries/stablehlo-opt out/01_stablehlo_clean.mlir --stablehlo-legalize-to-linalg -linalg-fuse-elementwise-ops --stablehlo-target-independent-optimization -o out/02_linalg_generic_tensor.mlir
=== GNERIC LINALG (TENSOR) MLIR ===
#map = affine_map<(d0) -> (d0)>
module @a_inference_add_fn_8__.1 attributes {mhlo.cross_program_prefetches = [], mhlo.input_output_alias = [], mhlo.is_dynamic = false, mhlo.use_auto_spmd_partitioning = false} {
  func.func @main(%arg0: tensor<2xf32>, %arg1: tensor<2xf32>) -> tensor<2xf32> {
    %0 = tensor.empty() : tensor<2xf32>
    %1 = linalg.generic {indexing_maps = [#map, #map, #map], iterator_types = ["parallel"]} ins(%arg0, %arg1 : tensor<2xf32>, tensor<2xf32>) outs(%0 : tensor<2xf32>) {
    ^bb0(%in: f32, %in_0: f32, %out: f32):
      %2 = arith.addf %in, %in_0 : f32
      linalg.yield %2 : f32
    } -> tensor<2xf32>
    return %1 : tensor<2xf32>
  }
}




In [9]:
# ============================================================
# 4.1 StableHLO → Linalg (Tensor)
# ============================================================
out2 = Path(OUT_DIR) / "02_linalg_specialized_tensor.mlir"

run([
    STABLEHLO_OPT,
    str(out2_1),
    "-linalg-fuse-elementwise-ops",  
    "--stablehlo-target-independent-optimization",
    "-o", str(out2),
])

print("=== LINALG (TENSOR) MLIR ===")
print(out2.read_text())

>> /workspace/PyTorchSim/Tensorflow/binaries/stablehlo-opt out/02_linalg_generic_tensor.mlir -linalg-fuse-elementwise-ops --stablehlo-target-independent-optimization -o out/02_linalg_specialized_tensor.mlir
=== LINALG (TENSOR) MLIR ===
#map = affine_map<(d0) -> (d0)>
module @a_inference_add_fn_8__.1 attributes {mhlo.cross_program_prefetches = [], mhlo.input_output_alias = [], mhlo.is_dynamic = false, mhlo.use_auto_spmd_partitioning = false} {
  func.func @main(%arg0: tensor<2xf32>, %arg1: tensor<2xf32>) -> tensor<2xf32> {
    %0 = tensor.empty() : tensor<2xf32>
    %1 = linalg.generic {indexing_maps = [#map, #map, #map], iterator_types = ["parallel"]} ins(%arg0, %arg1 : tensor<2xf32>, tensor<2xf32>) outs(%0 : tensor<2xf32>) {
    ^bb0(%in: f32, %in_0: f32, %out: f32):
      %2 = arith.addf %in, %in_0 : f32
      linalg.yield %2 : f32
    } -> tensor<2xf32>
    return %1 : tensor<2xf32>
  }
}




In [10]:
out6 = Path(OUT_DIR) / "06.mlir"

run([
    MLIR_OPT,
    str(out2),                      # bufferized input
    "-one-shot-bufferize=\"bufferize-function-boundaries\"",
    "-convert-bufferization-to-memref",
    "-convert-linalg-to-loops",
    # "-convert-vector-to-scf=full-unroll",
    # "-convert-index-to-llvm",
    # "-scf-for-loop-canonicalization",

    # "-convert-scf-to-cf",
    # "-convert-cf-to-llvm",

    # # "-convert-scf-to-cf",
    # "-lower-affine",
    # "-lower-vector-multi-reduction",


    # # # ---- memref / func / index ----
    # "-expand-strided-metadata",
    # "-finalize-memref-to-llvm",
    # "-convert-func-to-llvm",

    # # # ---- LLVM lowering ----
    # "-convert-arith-to-llvm",
    # "-convert-vector-to-llvm",
    # "-convert-math-to-llvm",
    
    # # # ---- cleanup ----
    # "-reconcile-unrealized-casts",

    "-o", str(out6),
])


print("=== LLVM DIALECT MLIR ===")
print(out6.read_text())


>> /workspace/PyTorchSim/Tensorflow/binaries/mlir-opt out/02_linalg_specialized_tensor.mlir -one-shot-bufferize="bufferize-function-boundaries" -convert-bufferization-to-memref -convert-linalg-to-loops -o out/06.mlir
=== LLVM DIALECT MLIR ===
module @a_inference_add_fn_8__.1 attributes {mhlo.cross_program_prefetches = [], mhlo.input_output_alias = [], mhlo.is_dynamic = false, mhlo.use_auto_spmd_partitioning = false} {
  func.func @main(%arg0: memref<2xf32, strided<[?], offset: ?>>, %arg1: memref<2xf32, strided<[?], offset: ?>>) -> memref<2xf32> {
    %c1 = arith.constant 1 : index
    %c2 = arith.constant 2 : index
    %c0 = arith.constant 0 : index
    %alloc = memref.alloc() {alignment = 64 : i64} : memref<2xf32>
    scf.for %arg2 = %c0 to %c2 step %c1 {
      %0 = memref.load %arg0[%arg2] : memref<2xf32, strided<[?], offset: ?>>
      %1 = memref.load %arg1[%arg2] : memref<2xf32, strided<[?], offset: ?>>
      %2 = arith.addf %0, %1 : f32
      memref.store %2, %alloc[%arg2] : memre

In [ ]:
import torch, os, sys, subprocess, re, io
from pathlib import Path
from contextlib import redirect_stdout
f = io.StringIO()
with redirect_stdout(f):
    base_dir = os.environ.get("TORCHSIM_DIR", "/workspace/PyTorchSim")
    sys.path.append(base_dir)
    from Scheduler.scheduler import PyTorchSimRunner
    device = PyTorchSimRunner.setup_device().custom_device()

    a = torch.tensor([1.0], dtype=torch.float32, device=device)
    b = torch.tensor([3.0, 4.0], dtype=torch.float32, device=device)

    def add(x, y):
        return torch.add(x, y)

    opt_fn = torch.compile(dynamic=False)(add)
    _ = opt_fn(a, b)

stdout = f.getvalue()
m = re.search(r"Wrapper Codegen Path = (.+)", stdout)
wrapper_path = Path(m.group(1).strip())
code = wrapper_path.read_text()
mlir_match = re.search(r"custom_async_compile\.mlir\(\s*'''(.*?)'''\s*,?",code,re.DOTALL,)
print(mlir_match.group(1))


RuntimeError: Failed to load the backend extension: torch_openreg. You can disable extension auto-loading with TORCH_DEVICE_BACKEND_AUTOLOAD=0.